# Check OU hierarchy consistency between SNIS and FBP

Extracts SNIS data for a period, keeps the OUs that have data, and compares their
hierarchy against the FBP pyramid with an anti-join in both directions.

In [23]:
from datetime import datetime
from pathlib import Path

import polars as pl
import yaml
from openhexa.sdk.workspaces.connection import DHIS2Connection
from openhexa.toolbox.dhis2 import DHIS2
from openhexa.toolbox.dhis2.dataframe import get_organisation_units

import config
from api import get_datasets_as_dict
from dates import isodate_to_period_type
from openhexa.toolbox.dhis2.periods import get_range

In [24]:
WORKSPACE_DIR = Path(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/transfer_snis_fbp/workspace"
)
WORKSPACE_YAML = Path(
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/burundi/transfer_snis_fbp/workspace.yaml"
)
SOURCE_CONNECTION = "snis"
TARGET_CONNECTION = "pbf-burundi"
MAPPING_FILE = "Mapping SNIS-FBP - Mapping_new_CDS.csv"
START_DATE = "2026-01-01"
END_DATE = "2026-07-30"

# set to an existing run folder name to reuse all_extracted_data.parquet instead of extracting
REUSE_RUN = None

In [25]:
def get_dhis2_client(connection: DHIS2Connection, use_cache: bool) -> DHIS2:
    cache_dir = Path(".cache") if use_cache else None
    return DHIS2(connection=connection, cache_dir=cache_dir)


def validate_connection(dhis2: DHIS2, name: str = "DHIS2") -> None:
    try:
        dhis2.meta.system_info()
        print(f"OK {name} connection successful: {dhis2.api.url}")
    except Exception as e:
        print(f"ERROR {name} connection failed: {e!s}")
        raise


def read_csv(file_path: Path) -> pl.DataFrame:
    try:
        return pl.read_csv(file_path)
    except pl.exceptions.ComputeError:
        print(f"UTF-8 decoding failed for {file_path}. Retrying with 'iso-8859-1'...")
        return pl.read_csv(file_path, encoding="iso-8859-1")


def get_periods_from_start_end_dates(start_date, end_date, period_type) -> list[str]:
    start_period = isodate_to_period_type(start_date, period_type)
    end_period = isodate_to_period_type(end_date, period_type)
    if str(start_period) == str(end_period):
        periods = [start_period]
    else:
        periods = get_range(start_period, end_period)
    if not periods:
        raise ValueError(
            f"No periods found between {start_date} and {end_date} for period type '{period_type}'."
        )
    print(f"Querying {len(periods)} periods: {periods[0]} to {periods[-1]}")
    return [str(p) for p in periods]


def extract_source_data(source_dhis2, dataset_id, periods_extraction, dataset_org_units):
    print(f"Extracting data from dataset {dataset_id}...")
    source_dhis2.data_value_sets.MAX_ORG_UNITS = config.max_org_unit_per_request
    source_dhis2.data_value_sets.MAX_PERIODS = config.max_periods_per_request
    result = source_dhis2.data_value_sets.get(
        datasets=[dataset_id], periods=periods_extraction, org_units=dataset_org_units
    )
    data_values_list = result if isinstance(result, list) else []
    if not data_values_list:
        print("No data values returned from API")
        return pl.DataFrame()

    data_values = (
        pl.DataFrame(data_values_list, infer_schema_length=None)
        .with_columns(pl.lit(dataset_id).alias("dataset_id_snis"))
        .select(
            [
                "dataset_id_snis",
                "dataElement",
                "orgUnit",
                "categoryOptionCombo",
                "attributeOptionCombo",
                "period",
                "value",
            ]
        )
        .rename(
            {
                "dataElement": "data_element_id",
                "orgUnit": "organisation_unit_id",
                "categoryOptionCombo": "category_option_combo_id",
                "attributeOptionCombo": "attribute_option_combo_id",
            }
        )
    ).with_columns(pl.col("value").cast(pl.Float64, strict=False))

    print(f"Extracted {len(data_values)} data values")
    print(f"  - Unique org units: {data_values['organisation_unit_id'].n_unique()}")
    return data_values


def extract_dataset_data(
    source_dhis2, dataset_id, datasets_source, start_date, end_date, output_dir
):
    period_type = datasets_source[dataset_id]["periodType"]
    print(f"Dataset {dataset_id}: period type {period_type}")
    periods = get_periods_from_start_end_dates(start_date, end_date, period_type)
    org_units = datasets_source[dataset_id]["organisation_units"]
    data = extract_source_data(source_dhis2, dataset_id, periods, org_units)
    if len(data) == 0:
        print(f"No data extracted for dataset {dataset_id}")
    else:
        data.write_parquet(output_dir / f"{dataset_id}.parquet")
    return data


def extract_all_dataset_data(
    source_dhis2, dataset_ids, datasets_source, start_date, end_date, output_dir
):
    print(f"Extracting data for {len(dataset_ids)} datasets...")
    extracted = [
        extract_dataset_data(source_dhis2, ds_id, datasets_source, start_date, end_date, output_dir)
        for ds_id in dataset_ids
    ]
    non_empty = [df for df in extracted if len(df) > 0]
    if not non_empty:
        print("No data extracted for any dataset")
    return pl.concat(non_empty) if non_empty else pl.DataFrame()

In [26]:
conf = yaml.safe_load(WORKSPACE_YAML.read_text())


def connection_from_yaml(name: str) -> DHIS2Connection:
    c = conf["connections"][name]
    return DHIS2Connection(url=c["url"], username=c["username"], password=c["password"])


snis = get_dhis2_client(connection_from_yaml(SOURCE_CONNECTION), config.use_cache)
fbp = get_dhis2_client(connection_from_yaml(TARGET_CONNECTION), config.use_cache)
validate_connection(snis, "SNIS")
validate_connection(fbp, "FBP")

OK SNIS connection successful: https://dhis.dsnis.bi/hmis/api
OK FBP connection successful: https://dhis2.minisante.gov.bi/api


## (a) SNIS data for the period

In [27]:
runs_dir = WORKSPACE_DIR / "pipelines" / "transfer_snis_fbp"

if REUSE_RUN:
    source_data = pl.read_parquet(runs_dir / REUSE_RUN / "all_extracted_data.parquet")
else:
    output_dir = runs_dir / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_dir.mkdir(parents=True, exist_ok=True)
    mapping_data = read_csv(WORKSPACE_DIR / config.pipeline_input / MAPPING_FILE)
    datasets_snis = get_datasets_as_dict(snis)
    dataset_ids = mapping_data["ds_id_snis"].drop_nulls().unique().to_list()
    source_data = extract_all_dataset_data(
        snis, dataset_ids, datasets_snis, START_DATE, END_DATE, output_dir
    )

ou_with_data = source_data["organisation_unit_id"].unique().to_list()
print(f"{len(source_data)} data values, {len(ou_with_data)} org units with data")

Extracting data for 1 datasets...
Dataset q2kgtbmnYvP: period type Monthly
Querying 7 periods: 202601 to 202607
Extracting data from dataset q2kgtbmnYvP...


Extracted 49649 data values
  - Unique org units: 826
49649 data values, 826 org units with data


## (b) & (c) Pyramids

In [28]:
pyramid_snis = get_organisation_units(snis)
pyramid_fbp = get_organisation_units(fbp)
print(f"SNIS pyramid: {len(pyramid_snis)} OUs, FBP pyramid: {len(pyramid_fbp)} OUs")

SNIS pyramid: 2860 OUs, FBP pyramid: 5028 OUs


In [29]:
snis_ous = pyramid_snis.filter(pl.col("id").is_in(ou_with_data))
missing_from_pyramid = set(ou_with_data) - set(snis_ous["id"].to_list())
if missing_from_pyramid:
    print(
        f"WARNING: {len(missing_from_pyramid)} OUs with data are absent from the SNIS pyramid: "
        f"{sorted(missing_from_pyramid)}"
    )
print(f"{len(snis_ous)} SNIS OUs with data retained for comparison")

826 SNIS OUs with data retained for comparison


## (d) Compare

In [30]:
def level_columns(df: pl.DataFrame) -> list[str]:
    levels = sorted(
        int(c.removeprefix("level_").removesuffix("_id"))
        for c in df.columns
        if c.startswith("level_") and c.endswith("_id")
    )
    return [f"level_{lvl}_{suffix}" for lvl in levels for suffix in ("id", "name")]


cols_snis = level_columns(pyramid_snis)
cols_fbp = level_columns(pyramid_fbp)

n_snis, n_fbp = len(cols_snis) // 2, len(cols_fbp) // 2
if n_snis != n_fbp:
    shared = min(n_snis, n_fbp)
    skipped = [f"level_{lvl}" for lvl in range(shared + 1, max(n_snis, n_fbp) + 1)]
    print(
        f"WARNING: SNIS has {n_snis} levels, FBP has {n_fbp}. "
        f"Comparing levels 1-{shared}; skipping {skipped}"
    )
    cols_snis = cols_snis[: shared * 2]

compare_cols = ["id", "name", "level", *cols_snis]
print(f"Comparing on: {compare_cols}")

Comparing on: ['id', 'name', 'level', 'level_1_id', 'level_1_name', 'level_2_id', 'level_2_name', 'level_3_id', 'level_3_name', 'level_4_id', 'level_4_name', 'level_5_id', 'level_5_name', 'level_6_id', 'level_6_name', 'level_7_id', 'level_7_name']


In [31]:
left = snis_ous.select(compare_cols)
right = pyramid_fbp.select(compare_cols)

only_in_snis = left.join(right, on=compare_cols, how="anti")
only_in_fbp = right.join(left.select("id"), on="id", how="semi").join(
    left, on=compare_cols, how="anti"
)

print(f"SNIS OUs with data whose row does not match FBP: {len(only_in_snis)}")
print(f"FBP rows (for those same OUs) that do not match SNIS: {len(only_in_fbp)}")

absent_in_fbp = set(left["id"].to_list()) - set(pyramid_fbp["id"].to_list())
print(f"SNIS OUs with data absent from the FBP pyramid entirely: {len(absent_in_fbp)}")
if absent_in_fbp:
    print(sorted(absent_in_fbp))

SNIS OUs with data whose row does not match FBP: 4
FBP rows (for those same OUs) that do not match SNIS: 1
SNIS OUs with data absent from the FBP pyramid entirely: 3
['ZoNxy1zz7JZ', 'g6Q9NZBrIYy', 'sHLJeiJB77o']


In [32]:
only_in_snis

id,name,level,level_1_id,level_1_name,level_2_id,level_2_name,level_3_id,level_3_name,level_4_id,level_4_name,level_5_id,level_5_name,level_6_id,level_6_name,level_7_id,level_7_name
str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""AFyf4SxabrU""","""CDS Bambo""",7,"""rdX5nU5lrcx""","""Burundi""","""JFo67UeKdx5""","""Bujumbura""","""RilEMAYMS3K""","""Muhuta""","""pEFuXCTlHgt""","""DS Bugarama""","""NS0t27zPXVR""","""Zone Ruteme""","""Ccbb3pxUuLg""","""AR Bambo""","""AFyf4SxabrU""","""CDS Bambo"""
"""sHLJeiJB77o""","""CDS Mubone(DS Mutimbuzi)""",7,"""rdX5nU5lrcx""","""Burundi""","""JFo67UeKdx5""","""Bujumbura""","""IpvHuHf5D0w""","""Ntahangwa""","""wpMEbjfJyq3""","""DS Mutimbuzi""","""WK1bBO1JicE""","""Zone Buterere""","""exo1G2BAkBK""","""AR Mubone""","""sHLJeiJB77o""","""CDS Mubone(DS Mutimbuzi)"""
"""ZoNxy1zz7JZ""","""HC Ntentamaza""",7,"""rdX5nU5lrcx""","""Burundi""","""nNMdItj5kTJ""","""Burunga""","""pP00S8YfoQb""","""Matana""","""kletPsdxuoA""","""DS Matana""","""Foan0MnJWvd""","""Zone Mugamba""","""PtYtXmVbbbi""","""AR Ntentamaza""","""ZoNxy1zz7JZ""","""HC Ntentamaza"""
"""g6Q9NZBrIYy""","""HC Tara""",7,"""rdX5nU5lrcx""","""Burundi""","""nNMdItj5kTJ""","""Burunga""","""pP00S8YfoQb""","""Matana""","""kletPsdxuoA""","""DS Matana""","""jkaxauu2K2Z""","""Zone Ruvumvu""","""rh6iKEXG3EU""","""AR Tara""","""g6Q9NZBrIYy""","""HC Tara"""


In [33]:
only_in_fbp

id,name,level,level_1_id,level_1_name,level_2_id,level_2_name,level_3_id,level_3_name,level_4_id,level_4_name,level_5_id,level_5_name,level_6_id,level_6_name,level_7_id,level_7_name
str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""AFyf4SxabrU""","""CDS Bambo""",7,"""rdX5nU5lrcx""","""Burundi""","""JFo67UeKdx5""","""Bujumbura""","""RilEMAYMS3K""","""Muhuta""","""pEFuXCTlHgt""","""DS Bugarama""","""NS0t27zPXVR""","""Zone Ruteme""","""I0dJ8WKPrPE""","""AR Bambo""","""AFyf4SxabrU""","""CDS Bambo"""
